# FNSPID — Task 1 exploratory data analysis

This notebook covers descriptive statistics, **temporal** news volume, **publisher** (and email-domain) patterns, and **textual** length / keyword signals using the financial news side of FNSPID.

**Prerequisite:** Place your news file(s) in `data/raw/`. Update `NEWS_PATH` below if the filename differs.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer

# Repo root: parent of notebooks/
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.text_utils import extract_publisher_domain, headline_char_length

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 4)

In [ ]:
# Set to your actual file (csv or parquet). Example names — adjust as needed.
RAW = ROOT / "data" / "raw"
candidates = sorted(RAW.glob("*.csv")) + sorted(RAW.glob("*.parquet"))
NEWS_PATH = candidates[0] if candidates else None

if NEWS_PATH is None:
    raise FileNotFoundError(
        f"No .csv or .parquet in {RAW}. Add FNSPID news data and re-run."
    )

print("Loading:", NEWS_PATH)
if NEWS_PATH.suffix.lower() == ".parquet":
    try:
        df = pd.read_parquet(NEWS_PATH)
    except ImportError as e:
        raise ImportError(
            "Reading Parquet requires pyarrow (or fastparquet). "
            "Install with: pip install pyarrow"
        ) from e
else:
    df = pd.read_csv(NEWS_PATH, low_memory=False)

df.head()

## Column harmonization

FNSPID fields: `headline`, `url`, `publisher`, `date`, `stock`. If your file uses different names, map them here.

In [ ]:
col_map = {
    "Headline": "headline",
    "headline": "headline",
    "Publisher": "publisher",
    "publisher": "publisher",
    "Date": "date",
    "date": "date",
    "stock": "stock",
    "Stock Symbol": "stock",
    "symbol": "stock",
}
df = df.rename(columns={k: v for k, v in col_map.items() if k in df.columns})

required = {"headline", "publisher", "date"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns after rename: {missing}. Got: {list(df.columns)}")

df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True)
df = df.dropna(subset=["date"])
df["headline"] = df["headline"].astype(str)
df["publisher"] = df["publisher"].astype(str)

df["headline_len"] = df["headline"].map(headline_char_length)
df["publisher_domain"] = df["publisher"].map(extract_publisher_domain)

df.info()

## 1. Descriptive statistics (headline length)

Distribution of headline character counts — longer titles may carry more entities or nuance (or noise).

In [ ]:
display(df["headline_len"].describe())

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(df["headline_len"], bins=50, kde=True, ax=ax)
ax.set_title("Headline character length distribution")
ax.set_xlabel("Characters")
plt.tight_layout()
plt.show()

## 2. Temporal analysis — volume per day and day-of-week

**Question:** Do spikes align with weekdays? Does volume drop on weekends?

In [ ]:
df["date_only"] = df["date"].dt.tz_convert(None).dt.normalize()
daily = df.groupby("date_only").size().rename("articles").reset_index()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily["date_only"], daily["articles"], linewidth=0.8)
ax.set_title("Articles per calendar day")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

df["dow"] = df["date"].dt.day_name()
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow_counts = df["dow"].value_counts().reindex(dow_order).fillna(0)

fig, ax = plt.subplots(figsize=(9, 4))
dow_counts.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Article count by day of week")
ax.set_ylabel("Articles")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

display(dow_counts.to_frame("count"))

### Time-of-day (if timestamps have intraday resolution)

UTC-normalized hour distribution.

In [ ]:
df["hour"] = df["date"].dt.hour
hourly = df.groupby("hour").size()

fig, ax = plt.subplots(figsize=(10, 4))
hourly.plot(kind="bar", ax=ax, color="darkorange")
ax.set_title("Articles by hour of day (UTC from parsed column)")
ax.set_xlabel("Hour")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 3. Publisher analysis — raw vs domain grouping

Email-style publishers are grouped by **domain** via `extract_publisher_domain` in `src/text_utils.py`.

In [ ]:
top_pub = df["publisher"].value_counts().head(15)
display(top_pub.to_frame("articles"))

top_dom = df["publisher_domain"].value_counts().head(15)
display(top_dom.to_frame("articles"))

fig, ax = plt.subplots(figsize=(9, 5))
top_dom.sort_values().plot(kind="barh", ax=ax, color="teal")
ax.set_title("Top publisher domains (or raw names)")
plt.tight_layout()
plt.show()

## 4. Keywords / n-grams (CountVectorizer)

Unigrams and bigrams for recurring financial phrasing (e.g. *beat*, *miss*, *surge*, *target*).

In [ ]:
headlines = df["headline"].fillna("").tolist()

vec = CountVectorizer(
    max_features=40,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
)
X = vec.fit_transform(headlines)
freq = np.asarray(X.sum(axis=0)).ravel()
terms = vec.get_feature_names_out()
top_idx = np.argsort(-freq)[:25]

top_terms = pd.DataFrame({"term": terms[top_idx], "count": freq[top_idx]})
display(top_terms)

fig, ax = plt.subplots(figsize=(8, 6))
sub = top_terms.sort_values("count")
ax.barh(sub["term"], sub["count"], color="slategray")
ax.set_title("Top headline terms / bigrams (CountVectorizer)")
plt.tight_layout()
plt.show()

verbs = ["surge", "plummet", "beat", "miss", "rise", "fall", "gain", "drop"]
low = df["headline"].str.lower()
# Substring counts (EDA); for stricter token boundaries use regex + word boundaries.
verb_hits = {v: int(low.str.contains(v, case=False, na=False).sum()) for v in verbs}
display(pd.Series(verb_hits, name="headline_matches").sort_values(ascending=False))

## 5. Takeaways (edit after you run on real data)

- **Volume:** Note any spikes and whether they coincide with earnings seasons or macro events (document in your report).
- **Weekends:** Expect lower counts if timestamps reflect US market-hours-centric publishing.
- **Publishers:** Domains consolidate contributor emails into organization-level counts.
- **Lexicon:** High-frequency n-grams hint at dominant narrative templates (guidance, M&A, ratings).

Next steps (Task 2+): merge with price data, indicators, sentiment scores, and correlation by trading day.